<a href="https://colab.research.google.com/github/aadhavjawahar-sys/Bank_Loan_Prediction/blob/main/PersonalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd #libary to work with data
import seaborn as sns #library for plotting
import numpy as np #library for working with numerical data
import matplotlib.pyplot as plt #Basis for seaborn
from sklearn.linear_model import LinearRegression

In [ ]:
df = pd.read_csv('/content/df1_loan.csv').dropna()
df.head()



,Unnamed: 0,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Total_Income
1,1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N,$6091.0
2,2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y,$3000.0
3,3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y,$4941.0
4,4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y,$6000.0
5,5,LP001011,Male,Yes,2,Graduate,Yes,5417,4196.0,267.0,360.0,1.0,Urban,Y,$9613.0


In [ ]:
feature_names=["Gender","Married","Education","LoanAmount","Loan_Amount_Term","Credit_History","Total_Income","Self_Employed"]
num_features = ["LoanAmount","Loan_Amount_Term","Credit_History","Total_Income"]
bin_features = ["Gender","Married","Education","Self_Employed"]
df['Total_Income'] = df['Total_Income'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['Total_Income'] = pd.to_numeric(df['Total_Income'], errors='coerce')
X = df[feature_names]
X.head()

,Gender,Married,Education,LoanAmount,Loan_Amount_Term,Credit_History,Total_Income,Self_Employed
1,Male,Yes,Graduate,128.0,360.0,1.0,6091.0,No
2,Male,Yes,Graduate,66.0,360.0,1.0,3000.0,Yes
3,Male,Yes,Not Graduate,120.0,360.0,1.0,4941.0,No
4,Male,No,Graduate,141.0,360.0,1.0,6000.0,No
5,Male,Yes,Graduate,267.0,360.0,1.0,9613.0,Yes


In [ ]:
y = df['Loan_Status']
y = y.map({"Y": 1, "N": 0})
y.head()

,Loan_Status
1,0
2,1
3,1
4,1
5,1


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('bin', OneHotEncoder(drop='first'), bin_features)
    ]
)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    #random_state=2026,
    stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_test_pred = model.predict(X_test_scaled)
y_train_pred = model.predict(X_train_scaled)
print("Coefficients:", model.coef_)

Coefficients: [[-0.46053517 -0.09890904  1.21289917 -0.07439602  0.51717731  0.40908876
  -0.56424465 -0.35350211]]


In [ ]:
scaler = preprocessor.named_transformers_['num']
print(scaler.mean_)
print(scaler.var_**0.5)

[1.41503425e+02 3.45123288e+02 8.56164384e-01 6.75442849e+03]
[7.11681637e+01 6.00214448e+01 3.50922971e-01 4.53974058e+03]


In [ ]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y_train, y_train_pred))
print(accuracy_score(y_test, y_test_pred))
print(X_test.iloc[[0]])
print(preprocessor.transform(X_test.iloc[[0]])[0])

0.8047945205479452
0.7857142857142857
    Gender Married Education  LoanAmount  Loan_Amount_Term  Credit_History  \
235   Male     Yes  Graduate       170.0             360.0             1.0   

     Total_Income Self_Employed  
235        6760.0            No  
[0.40041184 0.24785662 0.40987803 0.00122727 1.         1.
 0.         0.        ]


In [ ]:
import gradio as gr

def loan_status_check(gender, married, education, loanAmount, loanTerm, creditHistory, totalIncome, selfEmployed):
    input_df = pd.DataFrame(
        [
            {
                "Gender": gender,
                "Married": married,
                "Education": education,
                "LoanAmount": float(loan_amount),
                "Loan_Amount_Term": float(loan_term),
                "Credit_History": float(credit_history),
                "Total_Income": float(total_income),
                "Self_Employed": self_employed,
            }
        ]
    )
    dataPoint = preprocessor.transform(input_df)
    return model.predict(dataPoint)[0]
demo = gr.Interface(
    fn=loan_status_check,
    inputs=[
        gr.Dropdown(
            choices=["Male", "Female"], label="Gender", value="Male"
        ),
        gr.Dropdown(choices=["Yes", "No"], label="Married", value="Yes"),
        gr.Dropdown(
            choices=["Graduate", "Not Graduate"],
            label="Education",
            value="Graduate",
        ),

        gr.Number(label="Loan Amount", value=170.0),
        gr.Number(label="Loan Amount Term", value=360.0),
        gr.Number(label="Credit History (1.0 or 0.0)", value=1.0),
        gr.Number(label="Total Income", value=6760.0),

        gr.Dropdown(
            choices=["Yes", "No"], label="Self Employed", value="No"
        ),


        ],
    outputs=["text"],
    api_name="predict"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6413d4f6f6c878b10c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
